# SDB toolkit — satellite-derived bathymetry inputs for SEASTRONG (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alerovere/seastrong-field-tools/blob/main/SDB_toolkit/notebooks/seastrong_toolkit_colab.ipynb)
· Documentation: [seastrong-field-tools.readthedocs.io → SDB toolkit](https://seastrong-field-tools.readthedocs.io/en/latest/sdb_toolkit.html)

This notebook prepares the inputs for satellite-derived bathymetry (SDB) with the **Bathymetrix-AI** QGIS plugin, and
context layers for the WP1 seascape maps. The processing code is in the Python package `seastrong_toolkit` (folder
`SDB_toolkit/` of the repository); **all choices are made here, in the notebook.**

**How the notebook is organised.** Every step has four cells:

1. **Explanation** — what the step does, which data it uses, and how to choose each option.
2. **Options** — a Python dictionary with *every* option of the step, its default value and a comment. Change values
   here. A misspelt option name stops with a message (it is never ignored silently).
3. **Run** — runs the step. Results go to a Google Drive folder named after the step and a short code of its options:
   run it again with the same options and finished downloads/exports are re-used; change an option and only this step
   (and the steps that use its results) is recomputed, in a new folder.
4. **Report** — a plain-language summary of the results and, where something looks unusual, a **suggested option
   change** with the reason. Nothing is changed automatically: you decide, edit the options cell and run the step again.

| Step | What it gives you | Typical run time |
|---|---|---|
| 0 · Waves, wind, rain | hourly records at a point offshore since 1940 (ERA5) / 1980 (Copernicus Marine) | 5–20 min |
| 0b · Pre-screening | which months give the most usable Sentinel-2 images | 15–60 min (Earth Engine tables) |
| 1 · Sentinel-2 composite | the 12-band image for Bathymetrix-AI | 30 min – 3 h |
| 2 · Elevation | Copernicus DEM, FABDEM, DeltaDTM, masks | 10–30 min |
| 3 · Habitats | Allen Coral Atlas, Global Mangrove Watch, canopy height | 10–30 min |
| 4 · Water motion | turbidity / temperature tracers, currents (optional, slow) | hours |
| 5 · ICESat-2 | cleaned lidar depths (training / validation), depth limit of SDB, mangrove ground heights | 10–40 min |

Every output folder also contains a `README_*.md` (method, options used, limitations, references) and a
`manifest.json` (all options, software versions, Earth Engine task IDs, file checksums).

## Before you start: accounts and Colab Secrets

You need free accounts at **Google Earth Engine** (a Cloud project registered for Earth Engine), the **Copernicus Climate
Data Store** (accept the licences of *ERA5 hourly time-series data on single levels* and *ERA5 hourly data on single
levels*) and **Copernicus Marine**.

Store the log-in details as **Colab Secrets**: key icon in the left sidebar → *Add new secret* → switch on
*Notebook access* for each. They stay with your Google account and are never written into the notebook or the outputs.

| Secret | Value |
|---|---|
| `EE_PROJECT` | Earth Engine Cloud project ID |
| `EE_SERVICE_ACCOUNT_JSON` | *(optional)* service-account key, to log in to Earth Engine without the Google pop-up |
| `CDS_KEY` | Climate Data Store personal access token (your CDS profile page) |
| `CMEMS_USERNAME`, `CMEMS_PASSWORD` | Copernicus Marine log-in |

## Install the toolkit

The cell below downloads the repository and installs the package. Colab's own sign-in, Drive and Secrets depend on
exact versions of some Google packages (`google-auth`, `google-colab`, ...): a *constraints file* stops pip from
replacing them. Set `DEV_CLONE` to a Drive folder only if you want to edit the package code yourself.

In [ ]:
REPO = 'https://github.com/Alerovere/seastrong-field-tools.git'
SUBDIR = 'SDB_toolkit'        # folder of the package inside the repository
BRANCH = 'main'
DEV_CLONE = None              # e.g. '/content/drive/MyDrive/SEASTRONG/seastrong-field-tools' to edit the code

import subprocess, sys, re, os
frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.splitlines()
protect = re.compile(r'^(google-|googleapis-|grpcio|protobuf|proto-plus|earthengine-api|httplib2|ipykernel|ipython|'
                     r'jupyter|notebook|tornado|requests|urllib3)', re.I)
with open('/content/colab_constraints.txt', 'w') as f:
    f.write('\n'.join(l for l in frozen if '==' in l and protect.match(l)))
if DEV_CLONE:
    from google.colab import drive; drive.mount('/content/drive')
src = DEV_CLONE or '/content/seastrong-field-tools'
if not os.path.exists(src):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, src], check=True)
elif not DEV_CLONE:
    subprocess.run(['git', '-C', src, 'pull', '-q'], check=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *(['-e'] if DEV_CLONE else []), os.path.join(src, SUBDIR),
                    '-c', '/content/colab_constraints.txt'], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError('Installation failed (see above). Keep the constraints file: it protects Colab sign-in.')

In [ ]:
# Colab's autoreload still imports the 'imp' module removed in Python 3.12: use it only if it works.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload'); get_ipython().run_line_magic('autoreload', '2')
except Exception as e:
    print(f'autoreload not available ({e}); after editing the package code run st.reload() and the cells below again')
import seastrong_toolkit as st
print('seastrong_toolkit', st.__version__)

## Project: area of interest, map projection, names

- **Area of interest (AOI)**: either an Earth Engine table asset (`AOI_ASSET`) or a GeoJSON file in your Drive
  (`AOI_GEOJSON`, WGS84). Everything is clipped to it.
- **`CRS`**: the projected coordinate system of all rasters. Use the UTM zone of the site
  (southern Belize: `EPSG:32616`, UTM 16N; Seychelles (Mahé): `EPSG:32740`; Fiji (Viti Levu): `EPSG:32760`).
- **`PREFIX`** starts every file name; **`FOLDER_ROOT`** starts every Drive folder name.
- **`WAVE_REGION`**: the Copernicus Marine wave model: `'global'` everywhere, or `'med'` for the higher-resolution
  Mediterranean model (only for points in the Mediterranean).
- **`COPDEM_COLLECTION`**: the Copernicus DEM release used for the surface model and the land/water mask.
- **`RUN_TAG`**: leave empty. Write something (e.g. `'repeat1'`) only to force completely new output folders.

In [ ]:
PROJECT = dict(
    AOI_ASSET='projects/YOUR_PROJECT/assets/belize_south_aoi',   # Earth Engine table asset with the AOI polygon
    AOI_GEOJSON=None,          # or a WGS84 GeoJSON in Drive, e.g. '/content/drive/MyDrive/SEASTRONG/aoi_belize_south.geojson'
    CRS='EPSG:32616',          # UTM zone of the site
    GRID_SNAP=60,              # m; grid origin is a multiple of this, so 10/20/30/60 m outputs line up pixel-on-pixel
    PREFIX='Belize_south',     # start of every file name
    FOLDER_ROOT='SEASTRONG_Belize_south',      # start of every Drive folder name
    DRIVE_ROOT='/content/drive/MyDrive',       # where Google Drive is mounted
    COPDEM_COLLECTION='COPERNICUS/DEM/GLO30_2024_1',
    WAVE_REGION='global',      # 'global' or 'med'
    RUN_TAG='',
)
RUN = dict(WAIT_FOR_EXPORTS=True,     # wait for Earth Engine exports to finish (later steps need the files)
           WAIT_TIMEOUT_H=12,         # stop waiting after this many hours; run the step again later to continue
           DRIVE_SYNC_TIMEOUT_S=900)  # how long to wait for a finished export to appear in Drive

ctx = st.setup(PROJECT, RUN)          # Earth Engine log-in (Colab Secrets), Drive, AOI, export grid

## Step 0 — Waves, wind and rain at a point offshore

**What it does.** Downloads, from the providers' own services, hourly records at one point in deep water outside the
reef:

- **ERA5** (ECMWF reanalysis, Climate Data Store): significant wave height *Hs*, mean wave period and direction, 10 m
  wind and rain, **from 1940** to about 6 days ago. It uses the CDS *point time-series* service, so 86 years come in a
  few requests. ERA5 waves are on a 0.5° (~55 km) grid: the notebook picks the nearest *sea* cell to your point.
- **Copernicus Marine**: Hs, peak and mean period, direction from the global wave reanalysis (0.2°, 3-hourly, from 1980)
  plus the recent analysis (1/12°), or the Mediterranean model (1/24°, hourly, from 1985).

These records are used by the pre-screening (step 0b) and to drop rough-sea dates from the Sentinel-2 composite
(step 1), and are kept as a wave climate for later work.

**How to choose the options.**
- `POINT`: longitude, latitude in deep water **seaward of the reef**, within ~20–30 km of your AOI. Inside a lagoon the
  models do not represent waves. The report tells you how far the model cells are from your point.
- `MODE`: `'full'` keeps the whole record (recommended); `'screening'` downloads only `SCREENING_MONTHS` of
  `SCREENING_YEARS` (faster, but the pre-screening then has no full year to compare months).
- `SCREEN_SOURCE`: which wave record the Sentinel-2 date check uses: `'max'` (the larger of ERA5 and Copernicus Marine,
  cautious), `'era5'` or `'cmems'`.
- The other options rarely need changing; they control request sizes, caching and the provisional ERA5T data.

**ERA5 vs Copernicus Marine is a comparison of two models, not a validation:** the global Copernicus wave reanalysis
is driven by ERA5 winds, so agreement shows consistency, not accuracy.

In [ ]:
WAVES = dict(
    POINT=(-88.30, 16.10),     # lon, lat in deep water seaward of the reef
    MODE='full',               # 'full' = whole records; 'screening' = only SCREENING_MONTHS of SCREENING_YEARS
    ERA5_START='1940-01-01',   # first ERA5 date in 'full' mode
    CMEMS_START=None,          # None = start of the Copernicus Marine product (1980 global, 1985 Med)
    END=None,                  # None = latest available; or 'YYYY-MM-DD'
    CMEMS=True,                # download Copernicus Marine
    ERA5=True,                 # download ERA5
    ERA5_WIND=True,            # also ERA5 10 m wind and rain (needed by the pre-screening and the wind check)
    CMEMS_VARIABLES={'VHM0': 'hs', 'VTPK': 'tp', 'VTM10': 'tm10', 'VMDR': 'mwd'},   # Copernicus Marine name -> column
    ERA5_API='timeseries',     # 'timeseries' (fast, no peak period) or 'gridded' (slower, includes peak period)
    ERA5_TS_CHUNK_YEARS=20,    # years per time-series request (halved automatically if the CDS refuses)
    ERA5_LAG_DAYS=6,           # ERA5 is published ~5 days after real time
    ERA5T_REFRESH_DAYS=90,     # data younger than this are provisional (ERA5T) ...
    ERA5T_MAX_AGE_DAYS=7,      # ... and are downloaded again if the cached copy is older than this
    ERA5_SEARCH_DEG=0.75,      # box (deg) searched for the nearest ERA5 sea cell (max 1)
    ERA5_VARIABLES={'significant_height_of_combined_wind_waves_and_swell': 'hs', 'mean_wave_period': 'tm',
                    'peak_wave_period': 'tp', 'mean_wave_direction': 'mwd', '10m_u_component_of_wind': 'u10',
                    '10m_v_component_of_wind': 'v10', 'total_precipitation': 'rain_mm'},
    ERA5_CHUNK_MONTHS=3,       # 'gridded' API only: months per request
    CDS_RETRIES=3,
    SCREEN_SOURCE='max',       # wave record used by the Sentinel-2 date check: 'max', 'era5' or 'cmems'
    SCREENING_YEARS=[2020, 2026],   # 'screening' mode only
    SCREENING_MONTHS=[2, 4],        # 'screening' mode; also shaded in the wave figure
    PLOT_HS_THRESHOLD=1.5,     # Hs line drawn in the figure and used in the report (m)
)
ctx.set('waves', WAVES)

In [ ]:
st.waves.run(ctx)

**Look at the figure:** (a) the full records, (b) how the two models compare, (c) Hs by month — the months with the
lowest boxes are the calmest, (d) where waves come from, (e) how often Hs exceeds the threshold line.

In [ ]:
st.waves.report(ctx);

## Step 0b — Pre-screening: which months for the Sentinel-2 composite?

A good composite needs many images with **calm sea, no recent rain, clear sky, little sun glint and clear water**.
This step looks at all days of `YEARS` and answers: *in which months are such images most common, and how reliably
from year to year?*

**How a day and an image are judged.**
- A day is a **good sea-state day** when waves (max over the `WAVE_WINDOW_H` hours before the satellite overpass),
  wind (mean over `WIND_WINDOW_H` hours) and rain (sum of the `RAIN_DAYS` days before, from ERA5 and from CHIRPS) are
  all at or below their **site percentile** `PCTL` (computed over all months of all years) **and** below the fixed
  caps `ABS_CAPS`. With `PCTL` 75 a variable may not be among the worst 25 % of days at this site; the caps keep "calm"
  within the limits used later by the Sentinel-2 date check.
- A **usable Sentinel-2 image** is taken on a good day, with at least `CLEAR_MIN` of the AOI water cloud-free and
  turbidity (red reflectance of clear water) at or below the Sentinel-2 percentile `PCTL['turbidity']`.
  Landsat (since 1984) and Copernicus Marine ocean colour (since 2016) are shown in the figures for comparison.

**What you get.** A calendar of good days, a month-by-condition chart, usable images per month, and a ranking of 2-,
3- and 4-month windows by usable Sentinel-2 images per season (mean, spread between years, worst year). The report
compares the best window with `COMPARE_MONTHS` (e.g. the months you used so far).

**How to choose the options.**
- `YEARS`: a long period gives robust sea-state statistics; Sentinel-2 counts start in 2017 anyway.
- `PCTL`: higher = more permissive. If the report says very few days pass, raise the percentile of the variable that
  fails most often. Lower it if you want only the calmest conditions.
- `ORBIT`, `CS_THRESHOLD`, `PIXEL_MAX_SWIR`: use the same values as in the Sentinel-2 composite (step 1), otherwise the
  counts do not match the images that will actually be used.
- `OVERPASS_UTC`: local Sentinel-2 overpass time (Belize ~16:20 UTC; Seychelles ~06:40; Fiji ~22:30).

In [ ]:
PRESCREEN = dict(
    YEARS=[2000, 2026],        # years for sea-state and rain statistics (Sentinel-2 counts start in 2017)
    ORBIT=40,                  # Sentinel-2 relative orbit (None = all); same as in step 1
    CS_THRESHOLD=0.65,         # Cloud Score+ value above which a pixel is clear; same as in step 1
    PIXEL_MAX_SWIR=0.02,       # clear water with SWIR above this counts as sun glint; same as in step 1
    COMPARE_MONTHS=[2, 4],     # a window to compare with the ranking (first, last month)
    OVERPASS_UTC=16.3,         # typical Sentinel-2 overpass time (hours UTC)
    WAVE_WINDOW_H=48,          # waves: max Hs over these hours before the overpass (resuspension lasts)
    WIND_WINDOW_H=24,          # wind: mean speed over these hours before the overpass
    RAIN_DAYS=5,               # rain: sum over these whole days before the image (river plumes)
    RAIN_BUFFER=40000,         # m around the AOI for CHIRPS rain (include nearby catchments)
    PCTL={'hs': 75, 'wind': 75, 'rain_era5': 75, 'rain_chirps': 75, 'turbidity': 75},   # site percentiles (all months)
    ABS_CAPS={'hs': 1.5, 'wind': 6.0, 'rain_era5': 25.0, 'rain_chirps': 25.0},         # never above these (None = no cap)
    CLEAR_MIN=0.6,             # share of AOI water that must be cloud-free in a usable scene
    SENSORS=['S2', 'LANDSAT'], # scene statistics: 'S2', 'LANDSAT', 'HLS'
    LANDSAT_YEARS=[1984, 2026],
    OCEAN_COLOUR=True,         # Copernicus Marine ocean-colour transparency (shown in the heatmap)
    OC_DATASETS={'global': {'id': 'cmems_obs-oc_glo_bgc-transp_my_l3-olci-4km_P1D', 'variables': ['SPM', 'ZSD', 'KD490']},
                 'med': {'id': 'cmems_obs-oc_med_bgc-transp_my_l3-olci-300m_P1D', 'variables': ['KD490']}},
    OC_BUFFER_DEG=0.05,
    WINDOW_LENGTHS=[2, 3, 4],  # window lengths (months) to rank
    TOP_N=5,                   # windows listed per length
    STAT_SCALE=100,            # m; resolution of the scene statistics
)
ctx.set('prescreen', PRESCREEN)

In [ ]:
st.prescreen.run(ctx)

**Look at the figures:** the calendar shows whether good days cluster in the same months every year; the heatmap
shows *why* a month is bad (rain, wind, waves, clouds or turbidity); the bar chart ranks the windows. A window with a
high mean but a poor worst year is risky for a single-year composite.

In [ ]:
st.prescreen.report(ctx);

**Your decision:** write the chosen months into `MONTHS` of the Sentinel-2 options below (first, last month; a window
across the new year such as `[12, 2]` is allowed).

## Step 1 — Sentinel-2 composite for Bathymetrix-AI

**What it does.**
1. Collects all Sentinel-2 L2A images of `MONTHS` in `YEARS` over the AOI, from one relative orbit (`ORBIT`) so that
   viewing and sun-glint geometry are the same everywhere (mixing orbits creates seams).
2. Masks clouds per pixel (Cloud Score+ ≥ `CS_THRESHOLD`) and glint/whitecaps over water (SWIR > `PIXEL_MAX_SWIR`).
3. **Checks every date** and drops it if the water is glinty or hazy (`SCENE_MAX_NIR`), turbid (`SCENE_MAX_B5`), after
   heavy rain (`RAIN_MAX_MM` over `RAIN_DAYS` days), windy (`WIND_MAX_MS`) or with rough seas (`WAVE_MAX_HS`, from
   step 0). If a check has no data for a date, `MISSING_POLICY` decides whether the date is dropped or kept as
   unchecked. The report and figure show how many dates each check removed.
4. Takes a low percentile (`PERCENTILE`, 30) of the remaining images per pixel: haze and glint are *brighter*, so a
   low percentile removes them better than the median. Pixels with fewer than `MIN_VALID` clear observations are left
   empty (NoData).
5. **Matches the brightness of the MGRS tiles** (`GROUPS`, the first one is the reference): each tile is shifted by
   the median difference over water where it overlaps the tiles before it. Order the tiles so that each one overlaps
   water of an earlier one.

**Outputs.** `S2_…_12band_Float32.tif` (reflectance 0–1, NoData −9999, band order B1–B12 without B10 = Bathymetrix-AI
defaults), a QA raster (clear observations, source tile, invalid-reflectance flag), the blue temporal standard
deviation (high = unstable water), the screening table and figure, and the tile-offset table.

**How to tune.** Start with the defaults. If the report says a check removes a large share of dates, look at the
screening figure: if the distribution has a clear tail of bad dates, keep the threshold; if the threshold cuts through
the bulk of normal dates, raise it as suggested. If many pixels are NoData, add years or lower `MIN_VALID` (not below
3). In QGIS, look at bands 4-3-2 stretched 0–0.15 and check tile edges for seams.

In [ ]:
S2 = dict(
    YEARS=[2020, 2026],        # years of the composite
    MONTHS=[2, 4],             # first and last month (from the pre-screening)
    MAX_TILE_CLOUD=70,         # % cloud of a whole tile above which the image is skipped
    CS_THRESHOLD=0.65,         # Cloud Score+ clear-pixel threshold (higher = stricter)
    ORBIT=40,                  # relative orbit (None = all orbits; seams likely)
    PERCENTILE=30,             # per-pixel percentile (50 = median)
    MIN_VALID=5,               # minimum clear observations per pixel, else NoData
    SCALE=10,                  # m
    SENSITIVITY_MEDIAN=True,   # also export a median B1-B4 composite to test the depth limit (step 5)
    GROUP_BY='MGRS_TILE',      # tile matching by MGRS tile
    GROUPS=['16QCD', '16PCC', '16PBC', '16QBD'],   # tiles in matching order, reference first ([] = no matching)
    WATER_NDWI=0.2,            # water for matching: NDWI above this ...
    WATER_NIR=0.05,            # ... and NIR below this
    MATCH_BUFFER=10000,        # m around the AOI used to find overlap water
    OFFSET_SCALE=100,          # m; resolution for measuring offsets
    MIN_MATCH_PIXELS=50,       # fewer overlap water pixels = offset 'unsupported'
    OFFSET_MAX=0.01,           # offsets larger than this (reflectance) are flagged
    NO_WATER_FALLBACK='zero',  # unsupported offset: 'zero' (no shift, flagged), 'all' (use all overlap), 'error'
    GLINT_FILTER=True,         # mask glint pixels and drop glinty dates
    PIXEL_MAX_SWIR=0.02,       # per pixel: water with SWIR (B11) above this is masked
    SCENE_MAX_NIR=0.03,        # per date: median NIR (B8) of water above this = glint/haze -> date dropped
    TURBIDITY_FILTER=True,
    SCENE_MAX_B5=0.03,         # per date: median red-edge (B5) of water above this = turbid -> dropped
    RAIN_FILTER=True,
    RAIN_DAYS=5,               # CHIRPS rain over these whole days before the image ...
    RAIN_MAX_MM=25,            # ... above this (mm) -> dropped
    RAIN_BUFFER=40000,         # m around the AOI for the rain average
    WIND_FILTER=True,
    WIND_SOURCE='cds',         # 'cds' = ERA5 point series of step 0; 'gee' = ERA5 area mean in Earth Engine
    WIND_WINDOW_H=3,           # mean wind over these hours before the image ...
    WIND_MAX_MS=6,             # ... above this (m/s) -> dropped
    WIND_BUFFER=20000,         # 'gee' only: m around the AOI
    WAVE_FILTER=True,
    WAVE_WINDOW_H=6,           # max Hs over these hours before the image ...
    WAVE_MAX_HS=1.5,           # ... above this (m) -> dropped
    MISSING_POLICY={'waves': 'drop', 'wind': 'drop', 'rain': 'drop', 'water': 'drop'},   # 'drop' or 'keep' when a check has no data
    SCENE_STAT_SCALE=200,      # m; resolution of the per-date water statistics
    B1_FROM_B2=False,          # replace band 1 (coastal, 60 m) by a copy of B2 if B1 shows seams
    EXCLUDE_DATES=None,        # None = automatic checks; or a list of 'YYYY-MM-DD' to drop exactly these dates
    FIXED_OFFSETS=None,        # None = measure; or a dict {tile: {band: offset}} to force
)
ctx.set('s2', S2)

In [ ]:
st.s2.run(ctx)

In [ ]:
st.s2.report(ctx);

## Step 2 — Elevation (land and intertidal context)

Copernicus DEM (surface model: canopy tops in mangroves), FABDEM (bare earth by machine learning), DeltaDTM (coastal
bare earth corrected with lidar), DSM − FABDEM (≈ canopy height), the water-body mask and the edit mask, all in
metres above EGM2008 (not local mean sea level). FABDEM was made from an older Copernicus DEM release: where the edit
mask shows changes, DSM − FABDEM also contains release differences. ICESat-2 (step 5) compares these models with lidar
ground under mangroves.

In [ ]:
DEM = dict(
    SCALE=30,                  # m
)
ctx.set('dem', DEM)

In [ ]:
st.dem.run(ctx)
st.dem.report(ctx);

## Step 3 — Habitats

Allen Coral Atlas benthic and geomorphic classes (5 m), Global Mangrove Watch 2020 (10 m), mangrove canopy height,
and habitat areas computed at native resolution. Use them to split SDB errors by bottom type (dark seagrass often
comes out too deep, bright sand too shallow). The Coral Atlas maps only within its reef extent: `0` means *unmapped*.

In [ ]:
HABITATS = dict(
    AREA_SCALE_ACA=5,          # m; Allen Coral Atlas native resolution for area totals
    AREA_SCALE_GMW=10,         # m; Global Mangrove Watch native resolution
)
ctx.set('habitats', HABITATS)

In [ ]:
st.habitats.run(ctx)
st.habitats.report(ctx);

## Step 4 — Water-motion overview (optional, slow)

Maps of where turbid water and temperature anomalies go, on average and by condition (dry/wet season, wind sector,
after rain), from HLS or Sentinel-2 (recent) and Landsat (since 1984), plus regional HYCOM currents and ERA5 wind.
The turbidity values are a **relative tracer index**, not calibrated turbidity. Every conditioned map has its own
observation count and is masked where it has fewer than `MIN_SUPPORT` observations. Exports take hours: run this
step when the others are done.

In [ ]:
TRANSPORT = dict(
    TURB_SOURCE='HLS',         # recent tracer source: 'HLS' (30 m, Landsat + Sentinel-2) or 'S2' (20 m)
    A_YEARS=[2013, 2026],      # Part A years
    CS_THRESHOLD=0.65,         # Cloud Score+ (S2 source)
    C_YEARS=[1984, 2026],      # Part C: Landsat years
    C_PERIODS=[[1984, 1999], [2000, 2012], [2013, 2026]],   # Landsat periods compared
    USE_L7_SLC_OFF=True,       # include Landsat 7 after 2003 (gap stripes)
    D_YEARS=[1984, 2026],      # Part D: Landsat surface temperature years
    ST_VALID_C=[15, 40],       # plausible water temperature (°C)
    ST_MAX_UNC_K=1.5,          # max Landsat ST uncertainty (K)
    ST_MIN_CDIST_KM=1.0,       # min distance to clouds (km)
    ST_REF_GEOJSON=None,       # fixed reference area for temperature anomalies (None = all AOI water)
    ST_REF_MIN_COVER=0.5,      # share of the reference area that must be clear
    TURB_A=228.1, TURB_C=0.1641,   # tracer index coefficients (Dogliotti et al. 2015 form)
    PIXEL_MAX_SWIR=0.02,       # glint test
    PLUME_DT=5,                # anomaly above the per-pixel baseline that counts as a plume
    DRY_MONTHS=[2, 4], WET_MONTHS=[6, 11],
    RAIN_DAYS=5, RAIN_WET_MM=40, RAIN_DRY_MM=5, RAIN_BUFFER=40000,   # 'after rain' / 'no rain' conditions
    SECTORS={'N': [315, 45], 'E': [45, 135], 'S': [135, 225], 'W': [225, 315]},   # wind sectors (direction FROM)
    MIN_SUPPORT=5,             # minimum observations for a conditioned map
    CUR_YEARS=[2015, 2024],    # HYCOM currents period (HYCOM ends Sept 2024)
    CUR_BUFFER=60000,          # m around the AOI for currents and wind
    PARTS={'A': True, 'B': True, 'C': True, 'D': True},   # A recent tracer, B currents/wind, C Landsat tracer, D temperature
)
ctx.set('transport', TRANSPORT)

In [ ]:
st.transport.run(ctx)
st.transport.report(ctx);

## Step 5 — ICESat-2: training and validation depths, depth limit of SDB, mangrove ground

**What it does.**
1. Downloads ATL24 bathymetric lidar photons (SlideRule, no login) from `T0` to `T1`.
2. Removes unreliable photons with checks that do not compare tracks: not seabed or low confidence
   (`MIN_CONFIDENCE`), ATL24 quality flags or large uncertainty (`MAX_SIGMA_TVU`), implausible depth
   (`DEPTH_MIN`–`DEPTH_MAX`), on land or within `SHORE_BUFFER` of land, along-track spikes (`SPIKE_*`).
3. **Reserves validation tracks before anything is fitted**: whole ground tracks (RGTs) are set aside by a fixed rule and
   the choice is **locked** in a file in your Drive, so later runs never move validation tracks into training.
4. Flags photons that disagree with other *training* passes nearby (`XT_*`).
5. Finds the **depth limit of SDB** (*z_sat*): where the colour of the Sentinel-2 composite stops changing with depth.
   Deeper photons are not used for training; an *exploratory* map of optically shallow / deep / unknown water is made.
6. ATL08 ground and canopy heights inside mangroves, compared with the elevation models of step 2.

**Outputs for Bathymetrix-AI:** `…_ATL24_train.gpkg` (training) and `…_ATL24_valid.gpkg` (validation), depth field
`ortho_h` (negative below EGM2008). Keep echosounder data as the final, independent check.

**How to tune.** The report shows how many photons each check removes. If one check removes a lot, open
`…_ATL24_all_with_QC_flags.gpkg` in QGIS, look at the flagged photons (`f_*` columns) and loosen that option only if
they look like real seabed. If the depth limit is "unresolved", follow the report's suggestion and look at the figure.

In [ ]:
ICESAT2 = dict(
    T0='2018-10-13', T1=None,  # period (T1 None = today)
    SR_RETRIES=3,              # retries per year if SlideRule fails
    ON_MISSING_QC_FIELD='fail',   # 'fail' or 'quarantine' if ATL24 quality fields are missing
    MIN_CONFIDENCE=0.6,        # ATL24 seabed confidence
    MAX_SIGMA_TVU=1.0,         # m; max ATL24 total vertical uncertainty
    DEPTH_MIN=0.3, DEPTH_MAX=40.0,   # m; plausible depths
    SHORE_BUFFER=20.0,         # m from the land edge
    LAND_BUFFER_M=2000,        # m; land read beyond the AOI (coasts just outside count)
    SHORELINE_FILE=None,       # optional finer land polygons (GPKG/GeoJSON) for exact coast distances
    SPIKE_WINDOW=15, SPIKE_MIN=0.5,   # along-track running median (points) and minimum spike (m)
    VALID_SHARE=0.25,          # target share of photons in validation tracks
    MIN_RGT_TOTAL=4, MIN_RGT_VALID=1, MIN_PHOTONS_SIDE=50,   # minimum tracks / photons for a valid split
    HOLDOUT_LOCK=None,         # None = '<FOLDER_ROOT>_ICESat2_holdout_lock.json' in My Drive
    XT_RADIUS=30.0, XT_MIN=1.0, XT_REL=0.20, XT_MIN_PASSES=3, XT_BIN=10.0,   # cross-track check (m, m, fraction, passes, m)
    XT_APPLY_TO_VALID=False,   # True = also remove validation photons failing the cross-track check
    SDB_BIN=0.5, SDB_MIN_PER_BIN=30, SDB_MIN_RGT_PER_BIN=2,   # depth bins (m), photons and tracks per bin
    SDB_FLAT_TOL=0.05, SDB_FLAT_SPAN=3.0,   # 'flat' = change < 5 % of the range over 3 m
    SDB_MIN_PHOTONS=500, SDB_MIN_RGT=3, SDB_BOOT=200, SDB_MIN_RHO=0.3,   # data needed, bootstrap runs, min depth-colour correlation
    SDB_SENS={'SDB_BIN': [0.25, 0.5, 1.0], 'SDB_FLAT_TOL': [0.03, 0.05, 0.08], 'SDB_FLAT_SPAN': [2.0, 3.0, 4.0]},   # sensitivity tests
    COMPOSITE_TIF=None,        # None = composite of step 1; or a path to another 12-band composite
    MASK_MIN_AREA_PX=50,       # smaller patches of the optical map become 'unknown'
    RUN_ATL08=True, ATL08_LEN=100, MIN_GROUND_PH=5, MANGROVE_BUFFER_PX=1,   # ATL08 mangrove part
    GROUND_RANGE=[-3.0, 6.0],  # plausible mangrove ground (m, EGM2008)
)
ctx.set('icesat2', ICESAT2)

In [ ]:
st.icesat2.run(ctx)

In [ ]:
st.icesat2.report(ctx);

## Summary of everything run in this session
Task states and status of each step. A step marked *partial* or *unresolved* produced outputs with documented gaps:
read its README before using them.

In [ ]:
st.summary(ctx)